In [0]:
%pip install ax-platform
%pip install xgboost
%restart_python

In [0]:
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, PredictionErrorDisplay

from xgboost import XGBRegressor

from ax.service.managed_loop import optimize
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null
''').toPandas()

In [0]:
df_copy = df.copy()

high_missing_col = ['mainthread_garbageCollection','EXPERIMENTAL_TIME_TO_FIRST_BYTE','INTERACTION_TO_NEXT_PAINT']
df_copy = df_copy.drop(columns=high_missing_col)
df_copy=df_copy.dropna()

y = df_copy['largest-contentful-paint']

drop_cols = ['domain', 'url', 'performance_score', 'largest-contentful-paint','cumulative-layout-shift','first-contentful-paint','total-blocking-time','speed-index']
df_copy = df_copy.drop(columns=drop_cols)


In [0]:
X_encoded = pd.get_dummies(
    df_copy,
    columns=["strategy"],
    drop_first=False
)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [0]:
client2 = Client()

In [0]:
client2.configure_experiment(
    parameters=[
        RangeParameterConfig(
            name="max_depth",
            parameter_type="int",
            bounds=(4, 6),
        ),
        RangeParameterConfig(
            name="learning_rate",
            parameter_type="float",
            bounds=(0.01, 0.3),
        ),
        RangeParameterConfig(
            name="subsample",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="colsample_bytree",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="n_estimators",
            parameter_type="int",
            bounds=(100, 700),
        ),
        RangeParameterConfig(
            name="min_child_weight",
            parameter_type="int",
            bounds=(1, 10),
        ),
        RangeParameterConfig(
            name="gamma",
            parameter_type="float",
            bounds=(0, 1.0),
        ),
        RangeParameterConfig(
            name="reg_alpha",
            parameter_type="float",
            bounds=(0.0, 5.0),
        ),
        RangeParameterConfig(
            name="reg_lambda",
            parameter_type="float",
            bounds=(0.0, 10.0),
        ),
    ]
)
client2.configure_optimization(objective="-mae")

In [0]:
def evaluate_xgb(parameters):
    base_model = XGBRegressor(
        max_depth=int(parameters["max_depth"]),
        learning_rate=parameters["learning_rate"],
        n_estimators=int(parameters["n_estimators"]),
        subsample=parameters["subsample"],
        colsample_bytree=parameters["colsample_bytree"],
        min_child_weight=parameters["min_child_weight"],
        reg_alpha=parameters["reg_alpha"],
        reg_lambda=parameters["reg_lambda"],
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

    model = TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )

    mae = -scores.mean()

    return {"mae": mae}

In [0]:
for i in range(40):
    trails = client2.get_next_trials(max_trials=1)

    for trial_index, parameters in trails.items():
        result = evaluate_xgb(parameters)

        client2.complete_trial(
            trial_index = trial_index,
            raw_data = result,
        )

        print(f"Trial {trial_index}: MAE = {result["mae"]:.4f}")
        print(parameters)

In [0]:
best_parameters = client2.get_best_parameterization()
best_parameters

In [0]:
%skip
best_parameters = client.get_best_parameterization()
best_parameters

xgb = XGBRegressor(
    max_depth=8,
    learning_rate=0.041567487796869064,
    subsample=0.5,
    colsample_bytree=0.9992055956735079,
    n_estimators=700,
    min_child_weight=3,
    gamma=0.5591650100061931,
    reg_alpha=0.0,
    reg_lambda=0.1,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

In [0]:
# {'max_depth': 5,
#   'learning_rate': 0.115854151463143,
#   'subsample': 0.8475816888829094,
#   'colsample_bytree': 0.8444846514333358,
#   'n_estimators': 443,
#   'min_child_weight': 3,
#   'gamma': 0.0,
#   'reg_alpha': 1.136568199576757,
#   'reg_lambda': 10.0}
xgb = XGBRegressor(
    max_depth=5,
    learning_rate=0.115854151463143,
    subsample=0.8475816888829094,
    colsample_bytree=0.8444846514333358,
    n_estimators=443,
    min_child_weight=3,
    gamma=0.0,
    reg_alpha=1.136568199576757,
    reg_lambda=10.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model = TransformedTargetRegressor(
    regressor=xgb,
    func=np.log1p,
    inverse_func=np.expm1
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

print("XGBoost CV MAE scores:", -scores)
print("Mean CV MAE:", -scores.mean())
print("CV MAE std:", scores.std())

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

rmse = np.sqrt(mean_squared_error(y_test, test_pred))
r2 = r2_score(y_test, test_pred)

display = PredictionErrorDisplay(y_true=y_test, y_pred=test_pred)
display.plot()

print(f"Train MAE: {train_mae}")
print(f"Test MAE: {test_mae}")
print(f"Test R-squared: {r2}")
print(f"Test RMSE: {rmse}")

In [0]:
# %skip
# XGBoost CV MAE scores: [1289.95310138 1319.12043281 1407.37497604 1426.45027769 1280.54897802]
# Mean CV MAE: 1344.6895531871987
# CV MAE std: 60.62697545764086
# Train MAE: 1009.171185752865
# Test MAE: 1206.7429805873787
# Test R-squared: 0.9037666798926549
# Test RMSE: 2808.1619050730337